In [1]:
import torch

# 检查 CUDA 是否可用
print("CUDA Available:", torch.cuda.is_available())

# 查看当前 PyTorch 版本
print("PyTorch Version:", torch.__version__)

# 获取当前 CUDA 版本（如果可用）
if torch.cuda.is_available():
    print("CUDA Version:", torch.version.cuda)

    # 查看 GPU 设备数量
    print("GPU Count:", torch.cuda.device_count())

    # 获取当前默认 GPU 设备 ID
    print("Current GPU ID:", torch.cuda.current_device())

    # 获取当前 GPU 设备名称
    print("Current GPU Name:", torch.cuda.get_device_name(torch.cuda.current_device()))

import sys

# 显示python信息
print(sys.version)

import torch_geometric
import torch_sparse
print(torch_geometric.__version__)
print(torch_sparse.__version__)
from torch_geometric.data import DataLoader

import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.loader import NeighborLoader
from torch_geometric.data import Data
import networkx as nx
import numpy as np

# 生成一个示例图
def generate_graph(num_nodes=100, edge_prob=0.1):
    G = nx.erdos_renyi_graph(n=num_nodes, p=edge_prob)
    edge_index = torch.tensor(list(G.edges), dtype=torch.long).t().contiguous()
    x = torch.randn((num_nodes, 16))  # 随机特征
    y = torch.randint(0, 2, (num_nodes,))  # 随机二分类标签
    mask = torch.zeros(num_nodes, dtype=torch.bool)
    mask[:int(num_nodes * 0.1)] = True  # 10% 的节点有标签
    return Data(x=x, edge_index=edge_index, y=y, train_mask=mask)

# 定义GCN模型
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
    
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

# 生成数据
data = generate_graph()

# 使用 NeighborLoader 进行 k-跳邻域批次采样
loader = NeighborLoader(
    data,
    num_neighbors=[10, 10],  # 2层邻居，每层最多10个邻居
    batch_size=16,
    input_nodes=data.train_mask
)

# 初始化模型和优化器
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GCN(16, 32, 2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

# 训练模型
def train():
    model.train()
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)
        loss = F.cross_entropy(out[batch.train_mask], batch.y[batch.train_mask])
        loss.backward()
        optimizer.step()

# 运行训练
for epoch in range(100):
    train()
    print(f"Epoch {epoch+1}/100 completed")

print("Training completed.")


CUDA Available: True
PyTorch Version: 2.4.1
CUDA Version: 12.4
GPU Count: 1
Current GPU ID: 0
Current GPU Name: NVIDIA GeForce RTX 3060
3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:50:58) [GCC 12.3.0]


ModuleNotFoundError: No module named 'torch_geometric'